In [ ]:
# Preparation. SOURCE_RUN_ID is needed only the first time an episode is opened.
import json
from pathlib import Path

from drzavni_posao import MediaStore, MlflowTracker, Stage

EXPERIMENT_NAME = "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
AUDIO_PATHS = "/Volumes/luka_erceg/drzavni_posao/audios"
LABEL_DIR = Path("/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/labels")
SOURCE_RUN_ID = "b66150772bc043d091f7b4c9ff606ce0"
EPISODE_KEY = "56"
CHARACTERS = ["Djordje Cvarkov", "Dragan Torbica", "Boskic"]

tracker = MlflowTracker(EXPERIMENT_NAME)
label_path = LABEL_DIR / f"{EPISODE_KEY}.json"
gold = json.loads(label_path.read_text(encoding="utf-8")) if label_path.exists() else None
needs_source = gold is None or "pending" not in gold

if needs_source:
    config = tracker._configuration(SOURCE_RUN_ID)
    proposal = {key: config[key] for key in ("audio", "transcription")}
    if gold is not None and gold["proposal"] != proposal:
        raise ValueError("This episode was labeled from a different ASR configuration")
else:
    proposal = gold["proposal"]

media = MediaStore(AUDIO_PATHS, LABEL_DIR, **proposal["audio"])
episode = media.episode(EPISODE_KEY)
trim_start = proposal["audio"]["trim_start"]


def overlap(segment, segment_trim, label):
    return max(
        0,
        min(segment["end"] + segment_trim, label["end"])
        - max(segment["start"] + segment_trim, label["start"]),
    )


if needs_source:
    source = tracker.episode_run_states(SOURCE_RUN_ID)[EPISODE_KEY]
    tracker.restore_stage(episode, source.run_id, Stage.TRANSCRIPTION)
    if gold is None:
        gold = {
            "schema_version": 1,
            "episode_key": EPISODE_KEY,
            "timebase": "raw_audio",
            "proposal": proposal,
            "segments": [],
        }
    gold["pending"] = [
        segment
        for segment in episode.transcript.segments
        if not any(overlap(segment, trim_start, label) for label in gold["segments"])
    ]

for speaker, candidates in gold.pop("gallery_candidates", {}).items():
    for segment in gold["segments"]:
        if segment["speaker"] == speaker and any(
            segment["start"] == candidate["start"]
            and segment["end"] == candidate["end"]
            for candidate in candidates
        ):
            segment["gallery_candidate"] = True


def save():
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text(
        json.dumps(gold, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def gold_speaker(segment, candidate_trim_start=0):
    matches = [
        (overlap(segment, candidate_trim_start, label), label["speaker"])
        for label in gold["segments"]
    ]
    seconds, speaker = max(matches, key=lambda match: match[0], default=(0, None))
    return speaker if seconds else None


save()
audio = episode.audio()
print(
    f"{len(gold['segments'])} labeled, {len(gold['pending'])} pending, "
    f"{sum(segment.get('gallery_candidate', False) for segment in gold['segments'])} "
    "gallery candidates"
)

In [ ]:
# Labeling UI. Toggle ✓ before choosing a character to save a gallery candidate.
import ipywidgets as widgets
from IPython.display import Audio, clear_output, display

from drzavni_posao.constants import ANONYMOUS_LABEL, SR

queue = gold["pending"]
skipped = []
position = 0
gallery_selected = False
output = widgets.Output()
gallery = widgets.Button(description="✓", tooltip="Gallery candidate")
buttons = [widgets.Button(description=name) for name in [*CHARACTERS, ANONYMOUS_LABEL]]
skip = widgets.Button(description="Skip")
controls = [gallery, *buttons, skip]


def toggle_gallery(_):
    global gallery_selected
    gallery_selected = not gallery_selected
    gallery.description = "✓ ON" if gallery_selected else "✓"


def show():
    with output:
        clear_output(wait=True)
        if position >= len(queue):
            print("Pass complete. Rerun the preparation cell to revisit skipped segments.")
            for button in controls:
                button.disabled = True
            return
        segment = queue[position]
        print(f"{position + 1}/{len(queue)}  {segment['start']:.1f}–{segment['end']:.1f}s")
        print(segment.get("text", "").strip())
        start = max(0, int((segment["start"] - 0.25) * SR))
        end = int((segment["end"] + 0.25) * SR)
        display(Audio(audio[start:end], rate=SR, autoplay=True))


def advance(speaker=None):
    global position, gallery_selected
    segment = queue[position]
    if speaker is None:
        skipped.append(segment)
    else:
        label = {
            "start": segment["start"] + trim_start,
            "end": segment["end"] + trim_start,
            "speaker": speaker,
        }
        if gallery_selected and speaker != ANONYMOUS_LABEL:
            label["gallery_candidate"] = True
        gold["segments"].append(label)
    gallery_selected = False
    gallery.description = "✓"
    position += 1
    gold["pending"] = queue[position:] + skipped
    save()
    show()


gallery.on_click(toggle_gallery)
for button in buttons:
    button.on_click(lambda _, speaker=button.description: advance(speaker))
skip.on_click(lambda _: advance())
display(widgets.VBox([output, widgets.HBox(controls)]))
show()

In [ ]:
# Score a finished character-assignment parent run against this episode's gold labels.
import pandas as pd
from IPython.display import display

ASSIGNMENT_RUN_ID = ""
if not ASSIGNMENT_RUN_ID:
    raise ValueError("Set ASSIGNMENT_RUN_ID to a finished assignment parent run")

assignment_config = tracker._configuration(ASSIGNMENT_RUN_ID)
assignment = tracker.episode_run_states(ASSIGNMENT_RUN_ID)[EPISODE_KEY]
tracker.restore_stage(episode, assignment.run_id, Stage.CHARACTER_ASSIGNER)
assignment_trim_start = assignment_config["audio"]["trim_start"]

rows = []
for segment in episode.transcript.segments:
    expected = gold_speaker(segment, assignment_trim_start)
    if expected is not None:
        rows.append(
            {
                "start": segment["start"],
                "end": segment["end"],
                "gold": expected,
                "assigned": segment.get("speaker") or "[unassigned]",
                "text": segment.get("text", "").strip(),
            }
        )

results = pd.DataFrame(rows)
if results.empty:
    print("No assignment segments overlap the gold labels.")
else:
    print(f"accuracy: {(results['gold'] == results['assigned']).mean():.1%}")
    display(pd.crosstab(results["gold"], results["assigned"]))
    display(results[results["gold"] != results["assigned"]])

In [ ]:
# Build the expanded speaker gallery from every labeled episode.
gallery_path = Path("data/speaker_galleries/expanded.json")
gallery = {}

for path in sorted(LABEL_DIR.glob("[0-9]*.json")):
    labels = json.loads(path.read_text(encoding="utf-8"))
    for segment in labels.get("segments", []):
        if segment.get("gallery_candidate"):
            gallery.setdefault(segment["speaker"], []).append(
                {
                    "episode_key": labels["episode_key"],
                    "start": segment["start"],
                    "end": segment["end"],
                }
            )

for clips in gallery.values():
    clips.sort(key=lambda clip: (clip["episode_key"], clip["start"]))

gallery_path.parent.mkdir(parents=True, exist_ok=True)
gallery_path.write_text(
    json.dumps(gallery, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(
    f"Saved {sum(map(len, gallery.values()))} clips for {len(gallery)} characters "
    f"to {gallery_path}"
)
gallery